In [89]:
import importlib
import alkkagi_benchmark

importlib.reload(alkkagi_benchmark)
alkkagi_benchmark.demo2


<function alkkagi_benchmark.demo2(black_agent: kymnasium.agent.Agent, white_agent: kymnasium.agent.Agent)>

In [90]:
from alkkagi_benchmark import Agent
import gymnasium as gym
import numpy as np


def fight(
        train_agent: Agent,
        frozen_agent: Agent,
        track_round: int,
        win_threshold: float,
        max_round: int,
):
    env = gym.make('kymnasium/AlKkaGi-3x3-v0', render_mode='rgb_array', obs_type='custom', bgm=False)

    print(f'{train_agent.name_} vs. {frozen_agent.name_}: Started')

    train_agent.reset()
    frozen_agent.reset()

    history_win = np.zeros(shape=(track_round,), dtype='bool')
    history_reward = np.zeros(shape=(track_round,), dtype='float32')
    for i in range(max_round):
        obs, info = env.reset()
        done = False

        #train_agent.turn_ = i % 2
        #frozen_agent.turn_ = (i + 1) % 2
        train_agent.turn_ = 0
        frozen_agent.turn_ = 1
        steps = 0
        reward = 0.0
        total_reward = 0.0
        while not done:
            if obs['turn'] == train_agent.turn_:
                action = train_agent.act(obs, info)
            else:
                action = frozen_agent.act(obs, info)

            next_obs, _, terminated, truncated, info = env.step(action)
            steps += 1
            done = terminated or truncated or steps >= 30
            cur_self = obs['black'] if train_agent.turn_ == 0 else obs['white']
            cur_opponent = obs['white'] if train_agent.turn_ == 0 else obs['black']

            cur_self = np.sum(cur_self[:, 2] > 0)
            cur_opponent = np.sum(cur_opponent[:, 2] > 0)

            next_self = next_obs['black'] if train_agent.turn_ == 0 else next_obs['white']
            next_opponent = next_obs['white'] if train_agent.turn_ == 0 else next_obs['black']

            next_self = np.sum(next_self[:, 2] > 0)
            next_opponent = np.sum(next_opponent[:, 2] > 0)

            if obs['turn'] == train_agent.turn_:
                suicide = (cur_self - next_self) * -0.33
                knockout = (cur_opponent - next_opponent) * 0.33
                reward += (suicide + knockout)

            if done:
                reward += 1.0 if next_self > next_opponent else -1.0

            if done or obs['turn'] != train_agent.turn_:
                train_agent.store_reward(reward)
                total_reward += reward
                reward = 0.0
            '''
            if done:
                next_self = next_obs['black'] if train_agent.turn_ == 0 else next_obs['white']
                next_opponent = next_obs['white'] if train_agent.turn_ == 0 else next_obs['black']

                next_self = np.sum(next_self[:, 2] > 0)
                next_opponent = np.sum(next_opponent[:, 2] > 0)
                reward = 1.0 if next_self > next_opponent else -1.0
                train_agent.store_reward(reward)
                total_reward += reward
            elif obs['turn'] != train_agent.turn_:
                train_agent.store_reward(0.0)
            '''
            obs = next_obs

        black_count = np.sum(obs['black'][:, 2] > 0)
        white_count = np.sum(obs['white'][:, 2] > 0)

        if black_count > white_count:
            win = 0
        elif black_count < white_count:
            win = 1
        else:
            win = frozen_agent.turn_

        train_agent.train()
        train_agent.update_win_rate(win == train_agent.turn_)
        frozen_agent.update_win_rate(win == frozen_agent.turn_)

        history_win[i % track_round] = win == train_agent.turn_
        history_reward[i % track_round] = total_reward
        if i >= track_round and np.mean(history_win) >= win_threshold:
            break

        if i % 50 == 0:
            print(f'{train_agent.name_} vs. {frozen_agent.name_}: Round = {i}, Win Rate = {np.mean(history_win):.4f} / Reward = {np.mean(history_reward):.4f} / Actor Loss = {train_agent.actor_loss_:.4f} / Critic Loss = {train_agent.critic_loss_:.4f}')

    print(f'{train_agent.name_} vs. {frozen_agent.name_}: Completed; Win Rate = {np.mean(history_win):.4f}')

    return train_agent

In [92]:
from alkkagi_benchmark import Agent


agent1 = Agent(mode='rl', turn=0, deterministic=False, frozen=False)
agent2 = Agent(mode='random', turn=1, deterministic=False, frozen=True)

agent = fight(agent1, agent2, 50, 0.55, 5000)
agent.save('test')

ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Started
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Round = 0, Win Rate = 0.0000 / Reward = -0.0398 / Actor Loss = -0.1710 / Critic Loss = 0.0000
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Round = 50, Win Rate = 0.0400 / Reward = -1.8836 / Actor Loss = -0.0863 / Critic Loss = 0.0000
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Round = 100, Win Rate = 0.1000 / Reward = -1.7504 / Actor Loss = -0.0608 / Critic Loss = 0.0000
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Round = 150, Win Rate = 0.0800 / Reward = -1.8036 / Actor Loss = -0.0462 / Critic Loss = 0.0000
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 5e3b3f0e-c2fa-4710-9599-0903a0ef650a: Round = 200, Win Rate = 0.0400 / Reward = -1.8902 / Actor Loss = -0.0391 / Critic Loss = 0.0000
ee344df3-43e4-4120-b462-e2fce47d1b39 vs. 

KeyboardInterrupt: 

In [97]:
from tensorflow import keras


keras.ops.concatenate([np.array([[1, 2]]), np.array([[3, 4]])])

<tf.Tensor: shape=(2, 2), dtype=int64, numpy=
array([[1, 2],
       [3, 4]])>